# Module 7: Performance Attribution & Backtesting

**QuantVerse** — Quantitative Portfolio Intelligence System

---

## Objectives

1. **Walk-Forward Backtesting** — re-optimize on rolling windows, measure out-of-sample performance
2. **Rebalancing Analysis** — calendar vs threshold vs buy-and-hold, with transaction costs
3. **Brinson-Fachler Attribution** — allocation, selection, interaction effects
4. **30+ Performance Metrics** — Sharpe, Sortino, Calmar, Omega, Alpha, Beta, etc.
5. **Monthly Returns Table** — classic format for reporting
6. **Strategy Comparison** — equal weight, min var, max Sharpe, inverse vol, HRP

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, logging, sys, os, json

sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_palette('husl')
print('Setup complete.')

In [ ]:
# Load data
data_dir = '../data/processed'
daily_returns = pd.read_parquet(f'{data_dir}/returns_daily.parquet')
clean_prices = pd.read_parquet(f'{data_dir}/prices_clean.parquet')

with open(f'{data_dir}/asset_class_map.json', 'r') as f:
    class_map = json.load(f)

signal_tickers = [t for t, c in class_map.items() if c == 'signals']
investable = [t for t in daily_returns.columns if t not in signal_tickers]
returns = daily_returns[investable].dropna()

# Load portfolio weights
weights_path = f'{data_dir}/portfolio_weights.parquet'
if os.path.exists(weights_path):
    all_weights = pd.read_parquet(weights_path)
else:
    all_weights = pd.DataFrame({'Equal Weight': pd.Series(1/len(investable), index=investable)})

primary = 'Max Sharpe' if 'Max Sharpe' in all_weights.columns else all_weights.columns[0]
w_primary = all_weights[primary]
w_benchmark = pd.Series(1/len(investable), index=investable)

print(f'Assets: {len(investable)}, Obs: {len(returns)}')
print(f'Primary: {primary}')

## 1. Full Performance Report — Primary Strategy

In [ ]:
import json
from pathlib import Path

from project.backtest import PerformanceMetrics
from project.config import load_config

port_ret = pd.Series(returns.values @ w_primary.reindex(investable).fillna(0).values,
                      index=returns.index, name=primary)
bench_ret = pd.Series(returns.values @ w_benchmark.values,
                       index=returns.index, name='Equal Weight')

metadata_path = Path('data/processed/run_metadata.json')
risk_free_rate = json.loads(metadata_path.read_text(encoding='utf-8'))['risk_free_rate'] if metadata_path.exists() else load_config('configs/base.yaml').pipeline_kwargs()['fallback_risk_free_rate']
pm = PerformanceMetrics(port_ret, risk_free_rate=risk_free_rate)
report = pm.full_report(benchmark=bench_ret)

print(f'Full Performance Report — {primary}')
print('=' * 55)
for k, v in report.items():
    if isinstance(v, float):
        if 'Ratio' in k or 'Alpha' in k or 'Beta' in k:
            print(f'  {k:30s} {v:>10.4f}')
        elif 'Rate' in k or 'Months' in k or '%' in k:
            print(f'  {k:30s} {v*100:>10.2f}%')
        else:
            print(f'  {k:30s} {v*100:>10.2f}%')
    else:
        print(f'  {k:30s} {v}')

In [ ]:
# Multi-strategy performance comparison
all_reports = {}
for strat in all_weights.columns:
    sr = pd.Series(returns.values @ all_weights[strat].reindex(investable).fillna(0).values,
                   index=returns.index)
    pm_s = PerformanceMetrics(sr, risk_free_rate=risk_free_rate)
    all_reports[strat] = pm_s.full_report(benchmark=bench_ret)

key_metrics = ['CAGR', 'Annualized Volatility', 'Sharpe Ratio', 'Sortino Ratio',
               'Calmar Ratio', 'Max Drawdown', 'VaR (5%)', 'CVaR (5%)',
               'Win Rate', 'Best Month', 'Worst Month']

comp = pd.DataFrame({name: {k: r[k] for k in key_metrics} for name, r in all_reports.items()}).T

# Format
pct_cols = ['CAGR', 'Annualized Volatility', 'Max Drawdown', 'VaR (5%)', 'CVaR (5%)',
            'Win Rate', 'Best Month', 'Worst Month']
for c in pct_cols:
    if c in comp.columns:
        comp[c] = comp[c] * 100

print('Strategy Performance Comparison:')
print('=' * 110)
print(comp.round(4).to_string())

## 2. Equity Curves

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

for strat in all_weights.columns:
    sr = pd.Series(returns.values @ all_weights[strat].reindex(investable).fillna(0).values,
                   index=returns.index)
    cum = (1 + sr).cumprod()
    ax.plot(cum.index, cum.values, lw=1.5, alpha=0.8, label=strat)

ax.set_ylabel('Cumulative Return (1 = starting)', fontsize=13)
ax.set_title('Equity Curves — All Strategies', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1f}'))
plt.tight_layout()
plt.show()

## 3. Rebalancing Analysis

In [ ]:
from project.backtest import RebalancingEngine
from project.backtest.rebalancing import TransactionCosts

costs = TransactionCosts(proportional=0.001, spread=0.0005)
rebal = RebalancingEngine(returns, w_primary, costs=costs)
rebal_results = rebal.compare_all(threshold=0.05)

# Summary
rebal_summary = {}
for name, r in rebal_results.items():
    pm_r = PerformanceMetrics(r['portfolio_returns'])
    rebal_summary[name] = {
        'CAGR_%': pm_r.cagr() * 100,
        'Vol_%': pm_r.annualized_volatility() * 100,
        'Sharpe': pm_r.sharpe_ratio(),
        'Max_DD_%': pm_r.max_drawdown() * 100,
        'N_Rebalances': r['n_rebalances'],
        'Total_Turnover': r['total_turnover'],
        'Total_Cost_%': r['total_cost'] * 100,
    }

rebal_df = pd.DataFrame(rebal_summary).T
print(f'Rebalancing Comparison — {primary} (with transaction costs)')
print('=' * 90)
print(rebal_df.round(4).to_string())

In [ ]:
# Rebalancing equity curves
fig, ax = plt.subplots(figsize=(16, 7))

for name, r in rebal_results.items():
    ax.plot(r['portfolio_values'].index, r['portfolio_values'].values,
            lw=1.5, alpha=0.8, label=name)

ax.set_ylabel('Portfolio Value')
ax.set_title(f'Rebalancing Strategies — {primary}', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 4. Brinson-Fachler Attribution

In [ ]:
from project.backtest import PerformanceAttribution

attrib = PerformanceAttribution(returns, w_primary, w_benchmark, class_map)
bf = attrib.brinson_fachler()

print('Brinson-Fachler Attribution (vs Equal Weight Benchmark)')
print('=' * 100)
print(bf.round(3).to_string())

In [ ]:
# Attribution waterfall
bf_no_total = bf.drop('TOTAL')

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(bf_no_total))
width = 0.25

ax.bar(x - width, bf_no_total['Allocation_%'], width, label='Allocation', color='steelblue', edgecolor='white')
ax.bar(x, bf_no_total['Selection_%'], width, label='Selection', color='coral', edgecolor='white')
ax.bar(x + width, bf_no_total['Interaction_%'], width, label='Interaction', color='seagreen', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(bf_no_total.index, rotation=25, ha='right')
ax.set_ylabel('Active Return Contribution (%)')
ax.set_title('Brinson-Fachler Attribution by Asset Class', fontsize=14, fontweight='bold')
ax.legend()
ax.axhline(y=0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Asset-level contribution
asset_contrib = attrib.asset_contribution()

fig, ax = plt.subplots(figsize=(14, 8))
top = asset_contrib.head(10)
bottom = asset_contrib.tail(10)
plot_data = pd.concat([top, bottom]).drop_duplicates()

colors = ['green' if v > 0 else 'red' for v in plot_data['Active_Contribution_%']]
plot_data['Active_Contribution_%'].plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('Active Contribution (%)')
ax.set_title('Top & Bottom Active Return Contributors', fontsize=13, fontweight='bold')
ax.axvline(x=0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

## 5. Monthly Returns Table

In [ ]:
monthly = attrib.monthly_returns()
print(f'Monthly Returns (%) — {primary}')
print('=' * 100)

# Color formatting
styled = monthly.style.format('{:.2f}', na_rep='—') \
    .background_gradient(cmap='RdYlGn', axis=None, vmin=-10, vmax=10)
styled

## 6. Tracking Error Analysis

In [ ]:
te = attrib.tracking_error_analysis()
print(f'Tracking Error Analysis — {primary} vs Equal Weight')
print(f"  Annualized Tracking Error: {te['tracking_error']*100:.2f}%")
print(f"  Information Ratio:         {te['information_ratio']:.4f}")
print(f"  Active Return (ann):       {te['active_return']*100:.2f}%")

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(te['rolling_te'].index, te['rolling_te'] * 100, color='steelblue', lw=1.2)
axes[0].set_ylabel('Tracking Error (%)')
axes[0].set_title('Rolling 3-Month Tracking Error', fontweight='bold')

cum_active = te['active_returns'].cumsum() * 100
axes[1].fill_between(cum_active.index, cum_active.values, 0,
                     where=cum_active > 0, color='green', alpha=0.3)
axes[1].fill_between(cum_active.index, cum_active.values, 0,
                     where=cum_active < 0, color='red', alpha=0.3)
axes[1].plot(cum_active.index, cum_active.values, color='navy', lw=1)
axes[1].set_ylabel('Cumulative Active Return (%)')
axes[1].set_title('Cumulative Active Returns vs Benchmark', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Walk-Forward Backtest

In [ ]:
from project.backtest import PortfolioBacktester
from project.backtest.rebalancing import TransactionCosts

bt = PortfolioBacktester(returns, class_map, costs=TransactionCosts(proportional=0.001, spread=0.0005))

# Run all strategies with walk-forward (2-year train, quarterly rebal)
wf_results = bt.run_all_strategies(train_window=504, rebal_frequency=63)

# Summary
wf_summary = {}
for name, r in wf_results.items():
    m = r['metrics']
    wf_summary[name] = {
        'CAGR_%': m['CAGR'] * 100,
        'Vol_%': m['Annualized Volatility'] * 100,
        'Sharpe': m['Sharpe Ratio'],
        'Sortino': m['Sortino Ratio'],
        'Max_DD_%': m['Max Drawdown'] * 100,
        'Calmar': m['Calmar Ratio'],
        'N_Rebal': r['n_rebalances'],
        'Total_Turnover': r['total_turnover'],
        'Cost_Drag_%': r.get('annualized_cost_drag_%', 0),
    }

wf_df = pd.DataFrame(wf_summary).T
print('Walk-Forward Backtest Results (2yr train, quarterly rebal, NET of costs)')
print('=' * 100)
print(wf_df.round(4).to_string())

In [ ]:
# Walk-forward equity curves
fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [2, 1]}, sharex=True)

colors = plt.cm.Set2(np.linspace(0, 1, len(wf_results)))
for idx, (name, r) in enumerate(wf_results.items()):
    axes[0].plot(r['values'].index, r['values'].values, lw=1.5, color=colors[idx],
                alpha=0.8, label=f"{name} (SR={r['metrics']['Sharpe Ratio']:.2f})")

    dd = r['values'] / r['values'].cummax() - 1
    axes[1].plot(dd.index, dd.values * 100, lw=1, color=colors[idx], alpha=0.7)

axes[0].set_ylabel('Portfolio Value')
axes[0].set_title('Walk-Forward Backtest — Equity Curves', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_yscale('log')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1f}'))

axes[1].set_ylabel('Drawdown (%)')
axes[1].set_title('Drawdowns', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Rolling Sharpe Ratio

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

for name, r in wf_results.items():
    rolling_sharpe = r['returns'].rolling(126).apply(
        lambda x: x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else 0
    )
    ax.plot(rolling_sharpe.index, rolling_sharpe.values, lw=1.2, alpha=0.8, label=name)

ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axhline(y=1, color='green', linewidth=0.5, linestyle='--', alpha=0.5)
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Rolling 6-Month Sharpe Ratio', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 9. Rolling Asset Class Contribution

In [ ]:
rolling_ac = attrib.rolling_contribution(window=63)

fig, ax = plt.subplots(figsize=(16, 7))
rolling_ac.plot(ax=ax, linewidth=1.2)
ax.set_ylabel('Contribution (%)')
ax.set_title(f'Rolling 3-Month Return Contribution by Asset Class — {primary}',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 10. Export Backtest Results

In [ ]:
# Export walk-forward returns
wf_returns_df = pd.DataFrame({name: r['returns'] for name, r in wf_results.items()})
wf_returns_df.to_parquet(f'{data_dir}/backtest_returns.parquet')

# Export summary
wf_df.to_parquet(f'{data_dir}/backtest_summary.parquet')
# In-sample vs Out-of-sample comparison
print('\n=== In-Sample vs Out-of-Sample Performance ===')
is_sharpes = {}
for strat in all_weights.columns:
    sr = pd.Series(returns.values @ all_weights[strat].reindex(investable).fillna(0).values,
                   index=returns.index)
    pm_is = PerformanceMetrics(sr, risk_free_rate=risk_free_rate)
    is_sharpes[strat] = pm_is.sharpe_ratio()

if len(wf_results) > 0:
    for name, r in wf_results.items():
        oos_sharpe = r['metrics']['Sharpe Ratio']
        is_sharpe = is_sharpes.get(name, 0)
        delta = oos_sharpe - is_sharpe
        print(f"  {name:20s}  IS Sharpe={is_sharpe:.3f}  OOS Sharpe={oos_sharpe:.3f}  Δ={delta:+.3f}")

print('\nBacktest results exported.')


---

## Key Takeaways from Module 7

1. **Walk-forward reveals reality** — in-sample optimization ≠ out-of-sample performance
2. **Rebalancing frequency matters** — quarterly is often the sweet spot (cost vs drift)
3. **Transaction costs erode returns** — high-turnover strategies suffer disproportionately
4. **Attribution shows where value comes from** — allocation vs selection effects
5. **Equal weight is hard to beat** — a consistent finding across rolling periods
6. **Simpler strategies (Inv Vol, HRP) often outperform** complex optimizations out-of-sample

---

**QuantVerse Modules 1–7 Complete.**